# Assignment 2:
## Embeddings, Sequence Labeling, and Topic Classification

In [2]:
# Setup and imports
import os
import sys
import re
import json
import random
import math
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from sklearn.manifold import TSNE
import torch
import torch.nn as nn
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Device setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Constants
UNK = '<UNK>'
PAD = '<PAD>'
CLS = '<CLS>'
MAX_VOCAB = 10000
MAX_LEN = 256
CONTENT_LEN = MAX_LEN - 1

# Topic setup
TOPICS = ['Politics', 'Sports', 'Economy', 'International', 'Health & Society']
topic2id = {t: i for i, t in enumerate(TOPICS)}
id2topic = {i: t for t, i in topic2id.items()}

# Directory setup
DATA_DIR = Path('data')
MODEL_DIR = Path('models')
EMB_DIR = Path('embeddings')
PLOT_DIR = Path('plots')
OUT_DIR = Path('outputs')

for d in [DATA_DIR, MODEL_DIR, EMB_DIR, PLOT_DIR, OUT_DIR]:
    d.mkdir(exist_ok=True)

print('Setup complete')

Device: cpu
Setup complete


In [3]:
# Parse articles from cleaned.txt
def tokenize(text):
    text = text.replace('\u200e', '')
    return re.findall(r'\S+', text.lower())

def infer_topic(title, text, url):
    title_lower = title.lower()
    text_lower = text.lower()
    combined = title_lower + ' ' + text_lower
    
    politics_kw = {'وزیر', 'پارلیمان', 'حکومت', 'صدر', 'الیکشن', 'پیپلز', 'پاکستان', 'سیاست', 'شریف', 'بھٹو', 'عمران'}
    sports_kw = {'کریکٹ', 'فٹ بال', 'کھیل', 'ٹیم', 'میچ', 'بازی', 'پی سی بی', 'پاکستان', 'بیٹنگ', 'باؤلنگ'}
    economy_kw = {'روپیہ', 'بینک', 'اقتصادی', 'حوالہ', 'سرمایہ', 'تاجر', 'کاروبار', 'منڈی', 'شیئر', 'ڈالر'}
    intl_kw = {'امریکہ', 'چین', 'بھارت', 'روس', 'یورپ', 'انگریز', 'بین الاقوامی', 'ملک', 'سفیر'}
    health_kw = {'صحت', 'ڈاکٹر', 'ہسپتال', 'علاج', 'بیماری', 'دوا', 'وبائی', 'کوویڈ', 'صحت عامہ'}
    
    scores = defaultdict(int)
    for kw in politics_kw:
        scores['Politics'] += combined.count(kw)
    for kw in sports_kw:
        scores['Sports'] += combined.count(kw)
    for kw in economy_kw:
        scores['Economy'] += combined.count(kw)
    for kw in intl_kw:
        scores['International'] += combined.count(kw)
    for kw in health_kw:
        scores['Health & Society'] += combined.count(kw)
    
    if not scores:
        return 'Politics'
    return max(scores, key=scores.get)

def parse_articles(path):
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    articles = []
    blocks = content.strip().split('\n---\n')
    
    for block in blocks:
        lines = block.strip().split('\n')
        if len(lines) < 3:
            continue
        
        title = lines[0].strip()
        url = lines[1].strip() if len(lines) > 1 else ''
        text = '\n'.join(lines[2:]).strip()
        
        if not text or len(text) < 50:
            continue
        
        topic = infer_topic(title, text, url)
        articles.append({
            'title': title,
            'url': url,
            'text': text,
            'topic': topic
        })
    
    return articles

articles = parse_articles('cleaned.txt')
print(f'Loaded {len(articles)} articles')
print('Topic distribution:')
print(pd.Series([a['topic'] for a in articles]).value_counts())

Loaded 1 articles
Topic distribution:
Politics    1
Name: count, dtype: int64


In [4]:
# Part 1: TF-IDF embeddings
texts = [a['text'] for a in articles]

vectorizer = TfidfVectorizer(max_features=MAX_VOCAB, lowercase=True, tokenizer=tokenize, min_df=1, max_df=1.0)
tfidf_matrix = vectorizer.fit_transform(texts)
tfidf_matrix = tfidf_matrix.toarray()

vocab = vectorizer.get_feature_names_out()
w2i_c1 = {w: i for i, w in enumerate(vocab)}
i2w_c1 = {i: w for w, i in w2i_c1.items()}

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Vocabulary size: {len(w2i_c1)}')

np.save(str(EMB_DIR / 'tfidf_matrix.npy'), tfidf_matrix)
with open(str(EMB_DIR / 'word2idx.json'), 'w', encoding='utf-8') as f:
    json.dump(w2i_c1, f, ensure_ascii=False)
print('Saved TF-IDF to embeddings/')

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


TF-IDF matrix shape: (1, 7840)
Vocabulary size: 7840
Saved TF-IDF to embeddings/
